# Fig. 8: halo MSLE against radial motion

(a) The per-halo MSLE of Form 1 (Table 3, read from `table3_fig6_generalised_forms`'s `output/forms_per_halo.csv`)
against $\langle v_r\rangle/\sigma_r$ of the bound set, in the proper frame $v_r + H r$ used by the bound
selection ($<0$ is net infall). (b) Halo 1's projected density. Also prints the correlations of the MSLE
with $\langle v_r\rangle/\sigma_r$, $\beta$ and $c/a$.

Needs `$AURIGA_DATA`. Writes `gnuplot/{mse_scatter,mse_scatter_labels,halo01_projected_grid}.dat` and
`gnuplot/_meta.gnu` (read by `gnuplot/figure8_combined.gnu`), and `output/fig8_halos.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports
from Empirical_DF_Construction.paper_figures.common import bound_particles, out_dir

GP = Path.cwd() / "gnuplot"
GP.mkdir(exist_ok=True)
MSLE = pd.read_csv(out_dir() / "forms_per_halo.csv", index_col="halo")["msle1"]   # Form 1, Table 3
print(f"mean MSLE (Form 1) = {MSLE.mean():.4f}")

## Kinematics of the bound set

In [ ]:
def kinematics(h):
    # <v_r>/sigma_r in the proper frame, beta, and c/a from the reduced inertia tensor
    d = bound_particles(h)
    x, v, m = d["x_kpc"], d["v_kms"], d["m_msun"]
    r = np.linalg.norm(x, axis=1)
    vr = np.einsum("ij,ij->i", x, v) / r
    H = 0.1 * float(d["h"]) * np.sqrt(float(d["Omega0"]) / float(d["a"]) ** 3 + float(d["OmegaL"]))  # km/s/kpc
    vr_p = vr + H * r
    vt2 = np.maximum(np.einsum("ij,ij->i", v, v) - vr ** 2, 0)
    w = m / r ** 2
    lam = np.linalg.eigvalsh(np.einsum("k,ki,kj->ij", w, x, x) / w.sum())
    return dict(proxy=vr_p.mean() / vr_p.std(), beta=1 - vt2.mean() / (2 * vr.var()),
                c_over_a=np.sqrt(lam[0] / lam[2]))


T = pd.DataFrame({h: kinematics(h) for h in MSLE.index}).T.assign(msle1=MSLE)
T.to_csv(out_dir() / "fig8_halos.csv", index_label="halo")
for c in ("proxy", "beta", "c_over_a"):
    rho, p = spearmanr(T[c], T.msle1)
    print(f"MSLE vs {c:8s}: Pearson r = {pearsonr(T[c], T.msle1)[0]:+.3f}, Spearman rho = {rho:+.3f} (p = {p:.3g})")

## Figure data

In [ ]:
# (a) scatter, with the five worst fits labelled
worst = T.msle1.nlargest(5).index
np.savetxt(GP / "mse_scatter.dat", np.column_stack([T.proxy, T.msle1, T.index]),
           fmt=["%.8g", "%.8g", "%d"], header="mean_vr_proper_over_sigma_r  halo_msle_form1  halo_id")
np.savetxt(GP / "mse_scatter_labels.dat", np.column_stack([T.proxy[worst], T.msle1[worst], worst]),
           fmt=["%.8g", "%.8g", "%d"], header="label_x label_y halo_id")

# (b) Halo 1's projected density, +-1000 kpc on a 512^2 grid
HALF, NB, VMIN, VMAX = 1000.0, 512, 5.0, 7.5
d1 = bound_particles(1)
x, m = d1["x_kpc"], d1["m_msun"]
k = np.all(np.abs(x) < HALF, axis=1)
edges = np.linspace(-HALF, HALF, NB + 1)
Sigma = np.histogram2d(x[k, 0], x[k, 1], bins=edges, weights=m[k])[0].T / (2 * HALF / NB) ** 2
logS = np.where(Sigma > 0, np.log10(np.maximum(Sigma, 1e-30)), VMIN)
mid = 0.5 * (edges[:-1] + edges[1:])
with (GP / "halo01_projected_grid.dat").open("w") as fp:
    fp.write("# x_kpc y_kpc log10_sigma\n")
    for j, yy in enumerate(mid):
        fp.writelines(f"{xx:.6g} {yy:.6g} {logS[j, i]:.6g}\n" for i, xx in enumerate(mid))
        fp.write("\n")
(GP / "_meta.gnu").write_text("\n".join([
    f"PROXY_MIN = {T.proxy.min():.8g}", f"PROXY_MAX = {T.proxy.max():.8g}",
    f"MSE_MIN = {T.msle1.min():.8g}", f"MSE_MAX = {T.msle1.max():.8g}",
    f"SIGMA_VMIN = {VMIN}", f"SIGMA_VMAX = {VMAX}", f"MAP_EXTENT = {HALF}", f"NBINS = {NB}"]) + "\n")

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.5))
a.scatter(T.proxy, T.msle1, s=40, edgecolors="k")
for h in worst:
    a.annotate(str(h), (T.proxy[h], T.msle1[h]), xytext=(4, 4), textcoords="offset points")
a.set(xlabel=r"$\langle v_r\rangle/\sigma_r$", ylabel="Halo MSLE (Form 1)")
b.imshow(np.where(Sigma > 0, logS, np.nan), extent=[-HALF, HALF, -HALF, HALF], origin="lower",
         cmap="hot", vmin=VMIN, vmax=VMAX)
b.set(xlabel="x [kpc]", ylabel="y [kpc]")
plt.show()